# Experiments on Evaluation of RAG system

In [ ]:
# Fetching and parsing QASPER
# pip install datasets
from datasets import load_dataset

print("Downloading QASPER sample...")
# Load only the validation split to save time/RAM
qasper = load_dataset("allenai/qasper", split="validation")

# Let's find a question that actually has a written answer (some are unanswerable)
sample_qa = None
for row in qasper:
    # QASPER nests questions and answers deeply. We need to parse it carefully.
    questions = row['qas']['question']
    answers = row['qas']['answers']
    
    for q_idx, q in enumerate(questions):
        # Check if the answer has free-form text
        ans_list = answers[q_idx]['answer']
        for ans in ans_list:
            if ans['free_form_answer']:
                sample_qa = {
                    "question": q,
                    "ground_truth": ans['free_form_answer']
                }
                break
        if sample_qa: break
    if sample_qa: break

print("\n--- Extracted QASPER Sample ---")
print(f"Question: {sample_qa['question']}")
print(f"Ground Truth: {sample_qa['ground_truth']}")

# Citation Accuracy (ALCE)

To implement ALCE, we must use an LLM to perform an NLI (Natural Language Inference) task: comparing a Premise (the text of the cited document) against a Hypothesis (the generated sentence) to check for Entailment.

In [ ]:

# Experiment Implementing ALCE Entailment Logic
import re
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Mock Data from Phase 4
contexts = [
    "Recurrent models inherently process data sequentially.", # Index 0 -> [Doc 1]
    "The Transformer architecture allows for massive parallelization.", # Index 1 -> [Doc 2]
    "Apples are a type of fruit." # Index 2 -> [Doc 3]
]

# The LLM hallucinated the citation for the second sentence (Doc 3 is about apples!)
generated_answer = "Transformers replaced RNNs because they allow parallelization [Doc 2]. Furthermore, they process text sequentially [Doc 3]."

# 1. Split the answer into sentences (naive split for experiment)
sentences = [s.strip() for s in re.split(r'(?<=\.)\s+', generated_answer) if s.strip()]

# 2. Initialize the Judge LLM using local Ollama (free, no API key needed)
judge_llm = ChatOllama(model="llama3", temperature=0.0)

def check_entailment(claim: str, cited_text: str) -> bool:
    """Uses an LLM to check if the cited_text supports the claim (ALCE methodology)."""
    prompt = f"""
    Determine if the following CLAIM is fully supported by the provided DOCUMENT.
    DOCUMENT: "{cited_text}"
    CLAIM: "{claim}"
    Answer strictly with "TRUE" if supported, or "FALSE" if not supported.
    """
    response = judge_llm.invoke([HumanMessage(content=prompt)])
    return "TRUE" in response.content.upper()

print("--- ALCE METRIC EXPERIMENT ---")
supported_sentences = 0
sentences_with_citations = 0

for sentence in sentences:
    print(f"\nAnalyzing Sentence: '{sentence}'")
    
    # Find all [Doc X] tags in this specific sentence
    citations = re.findall(r'\[Doc (\d+)\]', sentence)
    
    if citations:
        sentences_with_citations += 1
        # Reconstruct the text the LLM cited. 
        # Note: LLM uses 1-based indexing [Doc 1], Python uses 0-based (contexts[0])
        cited_text = " ".join([contexts[int(doc_idx) - 1] for doc_idx in citations if int(doc_idx) <= len(contexts)])
        
        # Check Entailment
        is_supported = check_entailment(sentence, cited_text)
        print(f"Cited Text: '{cited_text}'")
        print(f"Entailment (Is Supported?): {is_supported}")
        
        if is_supported:
            supported_sentences += 1
    else:
        print("No citations found in this sentence.")

# ALCE Math
precision = supported_sentences / sentences_with_citations if sentences_with_citations > 0 else 0.0
recall = supported_sentences / len(sentences) if len(sentences) > 0 else 0.0

print(f"\nALCE Citation Precision: {precision * 100}%") 
print(f"ALCE Citation Recall: {recall * 100}%") 


In [ ]:
# Experiment Citation Accuracy Logic
import re

# Mock outputs from our pipeline
num_retrieved_docs = 3
generated_answer = "The Transformer is highly parallelizable [Doc 1]. It was introduced by Vaswani [Doc 2]. It also uses magic [Doc 4]."

def calculate_citation_accuracy(answer: str, num_docs: int) -> float:
    """
    Programmatic check based on ALCE. 
    Checks if citations are formatted correctly and do not point to non-existent documents.
    """
    # Regex to find all instances of [Doc X]
    citations = re.findall(r'\[Doc (\d+)\]', answer)
    
    if not citations:
        return 0.0 # Heavy penalty for failing to cite anything
        
    valid_citations = 0
    for citation in citations:
        doc_id = int(citation)
        # Check if the LLM hallucinated a document number (e.g., citing [Doc 4] when only 3 exist)
        if 1 <= doc_id <= num_docs:
            valid_citations += 1
            
    # Return ratio of valid citations to total citations made
    return valid_citations / len(citations)

score = calculate_citation_accuracy(generated_answer, num_retrieved_docs)
citations_found = re.findall(r'\[Doc (\d+)\]', generated_answer) 
print(f"Citations found: {citations_found}")
print(f"Citation Accuracy Score: {score:.2f}") 
# Notice how [Doc 4] drops the score because num_docs is 3!

# RAGAS Evaluation Experiment
Testing the RAGAS metrics pipeline (`ContextPrecision`, `ContextRecall`, `Faithfulness`, `ResponseRelevancy`) locally with mock data before integrating into `evaluate_rag.py`.

In [1]:
import nest_asyncio
nest_asyncio.apply()  # Fix: RAGAS uses asyncio internally which conflicts with Jupyter's event loop

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig
# Import from ragas.metrics (classic API) — compatible with evaluate()
# ragas.metrics.collections uses a different base class incompatible with evaluate()
from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama, OllamaEmbeddings

# --- Mock Data (mirrors the schema of data/evaluation_dataset.csv) ---
mock_data = [
    {
        "question": "What is the vanishing gradient problem in RNNs?",
        "ground_truth": "The vanishing gradient problem occurs when gradients become very small during backpropagation, making it difficult to train long sequences.",
        "contexts": [
            "Traditional RNN has problems of vanishing gradient and exploding gradient when training long sequences. By adding a gated mechanism to RNN, LSTM effectively solves these problems.",
            "Residual connections in a way makes the model selective of the contributing layers and solves the problem of vanishing gradients."
        ],
        "answer": "The vanishing gradient problem in Recurrent Neural Networks refers to the issue where gradients tend to vanish during backpropagation, making it difficult to train long sequences [Doc 1]."
    },
    {
        "question": "Why does the Transformer use positional encodings?",
        "ground_truth": "Since the Transformer has no recurrence or convolution, positional encodings are added to inject information about the position of tokens in a sequence.",
        "contexts": [
            "Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens.",
            "The Transformer is the first transduction model relying entirely on self-attention to compute representations of its input and output."
        ],
        "answer": "The Transformer needs positional encodings because it has no recurrence or convolution, so position information must be injected explicitly [Doc 1]. This allows the model to understand the order of tokens in a sequence [Doc 1]."
    }
]

df = pd.DataFrame(mock_data)

# --- Initialize Local Models wrapped for RAGAS compatibility ---
ragas_llm = LangchainLLMWrapper(ChatOllama(model="llama3", temperature=0.0))
ragas_embeddings = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))

# --- Build RAGAS Dataset ---
eval_dataset = Dataset.from_pandas(df)

# --- Define Metrics ---
metrics = [
    ContextPrecision(),
    ContextRecall(),
    Faithfulness(),
    AnswerRelevancy()
]

# --- RunConfig: increase timeout for local CPU inference (default 180s is too short for llama3 on CPU) ---
run_config = RunConfig(
    timeout=600,      # 10 minutes per LLM call
    max_retries=2,
    max_wait=30,
)

# --- Run Evaluation ---
# batch_size=1: process one row at a time to avoid memory crash on CPU
print("Starting RAGAS evaluation with local Ollama models...")
ragas_result = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config,
    raise_exceptions=False,
    batch_size=1
)

# --- Display Results ---
ragas_df = ragas_result.to_pandas()
print("\n===== RAGAS RESULTS =====")
print(ragas_df.to_string())
print("\n--- Averages ---")
for col in ['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']:
    if col in ragas_df.columns:
        print(f"{col}: {ragas_df[col].mean():.4f}")


/home/nguegnang/anaconda3/envs/qasper-rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting RAGAS evaluation with local Ollama models...


Evaluating: 100%|██████████| 8/8 [30:43<00:00, 230.45s/it]



===== RAGAS RESULTS =====
                                           user_input                                                                                                                                                                                                                                                                                                                                            retrieved_contexts                                                                                                                                                                                                                             response                                                                                                                                                 reference  context_precision  context_recall  faithfulness  answer_relevancy
0     What is the vanishing gradient problem in RNNs?                                      [Traditional RNN has problem

# Merge Test
Testing the `pd.concat` merge fix for `evaluate_rag.py` — verifies that RAGAS metric columns align correctly with the original `question`/`answer` columns without relying on a key join.

In [2]:
import pandas as pd

# --- Inspect what ragas_result.to_pandas() actually contains ---
print("=== ragas_df columns ===")
print(ragas_df.columns.tolist())
print(f"\nragas_df shape: {ragas_df.shape}")
print(ragas_df.head())

print("\n=== df columns ===")
print(df.columns.tolist())

# --- Replicate the evaluate_rag.py merge fix ---
# pd.merge on 'question'/'answer' fails because RAGAS 0.4.x drops those columns from result df.
# Fix: align by row index using pd.concat instead.
ragas_scores = ragas_df.reset_index(drop=True)
base = df[['question', 'ground_truth', 'answer']].reset_index(drop=True)

# Drop any columns that already exist in base to avoid duplicates
cols_to_drop = [c for c in ['question', 'answer', 'ground_truth', 'contexts'] if c in ragas_scores.columns]
final_df = pd.concat([base, ragas_scores.drop(columns=cols_to_drop, errors='ignore')], axis=1)

print("\n=== Merged final_df columns ===")
print(final_df.columns.tolist())
print(f"\nfinal_df shape: {final_df.shape}")
print(final_df.to_string())

print("\n--- Averages ---")
for col in ['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']:
    if col in final_df.columns:
        print(f"{col}: {final_df[col].mean():.4f}")
    else:
        print(f"{col}: NOT FOUND (NaN rows expected — metric failed for all rows)")

=== ragas_df columns ===
['user_input', 'retrieved_contexts', 'response', 'reference', 'context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']

ragas_df shape: (2, 8)
                                          user_input  \
0    What is the vanishing gradient problem in RNNs?   
1  Why does the Transformer use positional encodi...   

                                  retrieved_contexts  \
0  [Traditional RNN has problems of vanishing gra...   
1  [Since our model contains no recurrence and no...   

                                            response  \
0  The vanishing gradient problem in Recurrent Ne...   
1  The Transformer needs positional encodings bec...   

                                           reference  context_precision  \
0  The vanishing gradient problem occurs when gra...                1.0   
1  Since the Transformer has no recurrence or con...                1.0   

   context_recall  faithfulness  answer_relevancy  
0             1.0          0.

In [1]:
9+7

16

In [ ]:
%load_ext autoreload
%autoreload 2